## Section 0: Load Data and Preprocessing
 - Similar to the pretrained model (batch size = 32)

In [ ]:
import torch
import numpy as np
import random
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # For deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(0)

In [ ]:
# =========================================================================================
# Block 0-1: Load Data
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader

# Read the waveforms from the CSV file "B_waveform[T].csv"
waveforms = np.loadtxt("77/B_waveform[T].csv", delimiter=",")
num_waveforms, seq_len = waveforms.shape
print(f"Number of waveforms: {num_waveforms}")
print(f"Sequence length: {seq_len}")

num_harmonics = 13

# Load additional input variables
frequencies = np.loadtxt("77/Frequency[Hz].csv", delimiter=",")
temperatures = np.loadtxt("77/Temperature[C].csv", delimiter=",")

if frequencies.ndim == 1:
    frequencies = frequencies.reshape(-1, 1)
if temperatures.ndim == 1:
    temperatures = temperatures.reshape(-1, 1)

# Compute harmonic magnitudes
harmonic_magnitudes = np.zeros((waveforms.shape[0], num_harmonics))
for i in range(waveforms.shape[0]):
    waveform = waveforms[i]
    freq = frequencies[i, 0]
    n = len(waveform)
    fft_result = np.fft.rfft(waveform)
    fft_magnitude = np.abs(fft_result) / n * 2
    sample_rate = freq * n
    freqs = np.fft.rfftfreq(n, d=1.0/(freq * n))
    for k in range(1, num_harmonics + 1):
        harmonic_freq = k * freq
        idx = np.argmin(np.abs(freqs - harmonic_freq))
        harmonic_magnitudes[i, k-1] = fft_magnitude[idx]

# Load output prediction variable
volumetric_losses = np.loadtxt("77/Volumetric_losses[Wm-3].csv", delimiter=",")
if volumetric_losses.ndim == 1:
    volumetric_losses = volumetric_losses.reshape(-1, 1)

# Prepare input features
X_full = np.hstack([harmonic_magnitudes, frequencies, temperatures])
y_full = volumetric_losses

# Downsample to 10%
epsilon = 1e-12
y_log_full = np.log(y_full.flatten() + epsilon)
num_bins = 20
y_bins = np.digitize(y_log_full, np.histogram(y_log_full, bins=num_bins)[1][1:-1])
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.9, random_state=42)
for down_idx, _ in sss.split(X_full, y_bins):
    X = X_full[down_idx]
    y = y_full[down_idx]

print(f"Downsampled X shape: {X.shape}")
print(f"Downsampled y shape: {y.shape}")

In [ ]:
# =========================================================================================
# Block 0-2: Data Partition and Preprocessing

# Partition the data: train/val/test (60/20/20)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Standardize input features using training set statistics
input_scaler = RobustScaler()
X_train_scaled = input_scaler.fit_transform(X_train)
X_val_scaled = input_scaler.transform(X_val)
X_test_scaled = input_scaler.transform(X_test)

# Apply log transform then StandardScaler scaling to outputs
output_scaler = StandardScaler()
y_train_log = np.log(y_train + epsilon)
y_val_log = np.log(y_val + epsilon)
y_test_log = np.log(y_test + epsilon)

y_train_scaled = output_scaler.fit_transform(y_train_log.reshape(-1, 1))
y_val_scaled = output_scaler.transform(y_val_log.reshape(-1, 1))
y_test_scaled = output_scaler.transform(y_test_log.reshape(-1, 1))

# Create PyTorch DataLoaders
train_set = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), torch.tensor(y_train_scaled, dtype=torch.float32))
val_set = TensorDataset(torch.tensor(X_val_scaled, dtype=torch.float32), torch.tensor(y_val_scaled, dtype=torch.float32))
test_set = TensorDataset(torch.tensor(X_test_scaled, dtype=torch.float32), torch.tensor(y_test_scaled, dtype=torch.float32))

batch_size = 32 # to be adjusted
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

print(f"Train set: {X_train_scaled.shape[0]} samples")
print(f"Val set: {X_val_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")

## Section 1: Define LoRA-enhanced Model
 - To be adjusted: activation functions

In [ ]:
# =========================================================================================
# Block 1-1: Define LoRA Linear Layer
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """
    LoRA-enhanced Linear layer.
    The layer computes: output = W @ x + scale * B @ A @ x
    where W is the frozen pretrained weight, and B @ A is the low-rank update.
    """
    def __init__(self, in_features, out_features, rank=8, lora_alpha=16, lora_dropout=0.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        
        # Trainable LoRA parameters
        self.lora_a = nn.Parameter(torch.zeros(rank, in_features))
        self.lora_b = nn.Parameter(torch.zeros(out_features, rank))
        self.scale = lora_alpha / rank
        
        # Initialization
        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))
        nn.init.zeros_(self.lora_b)
        
        if lora_dropout > 0:
            self.dropout = nn.Dropout(lora_dropout)
        else:
            self.dropout = nn.Identity()

    def forward(self, x):
        # x: (batch_size, in_features)
        # Apply LoRA: scale * (B @ A) @ x
        lora_out = torch.matmul(
            torch.matmul(x, self.lora_a.t()),  # x @ A^T = (batch_size, rank)
            self.lora_b.t()  # B^T = (rank, out_features)
        )  # (x @ A^T) @ B^T = (batch_size, out_features)
        return self.scale * lora_out


class FeedforwardNNWithLoRA(nn.Module):
    """
    FNN with LoRA layers.
    Combines original frozen weights with trainable LoRA updates.
    """
    def __init__(self, pretrained_model, rank=8, lora_alpha=16, lora_dropout=0.0):
        super().__init__()
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        
        # Store pretrained model layers and create LoRA layers
        self.lora_layers = nn.ModuleDict()
        self.pretrained_layers = nn.ModuleDict()
        
        # Extract Linear layers from the pretrained model's sequential
        layer_idx = 0
        for i, module in enumerate(pretrained_model.net.modules()):
            if isinstance(module, nn.Linear):
                # Freeze pretrained weights
                module.weight.requires_grad = False
                module.bias.requires_grad = False
                
                self.pretrained_layers[f"linear_{layer_idx}"] = module
                
                # Create LoRA layer
                self.lora_layers[f"lora_{layer_idx}"] = LoRALinear(
                    module.in_features,
                    module.out_features,
                    rank=rank,
                    lora_alpha=lora_alpha,
                    lora_dropout=lora_dropout
                )
                layer_idx += 1
        
        # Store activation functions
        self.activation = nn.SiLU() # to be adjusted
        self.num_lora_layers = layer_idx
        
        # Recreate the sequential model structure
        self._build_net(pretrained_model)
    
    def _build_net(self, pretrained_model):
        """Rebuild the network with LoRA layers integrated"""
        layers = []
        lora_idx = 0
        
        for module in pretrained_model.net.modules():
            if isinstance(module, nn.Linear):
                layers.append(self.pretrained_layers[f"linear_{lora_idx}"])
                lora_idx += 1
            elif isinstance(module, nn.SiLU):
                layers.append(nn.SiLU())
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        # Forward pass through the base network
        base_out = self.net(x)
        
        # Add LoRA contributions layer by layer
        current = x
        lora_contribution = 0
        
        layer_idx = 0
        for i, module in enumerate(self.pretrained_layers.values()):
            if layer_idx == 0:
                # First layer
                lora_update = self.lora_layers[f"lora_{layer_idx}"](x)
                lora_contribution = lora_update
                # Get intermediate for next layer
                current = self.activation(module(x) + lora_update)
            elif layer_idx < self.num_lora_layers - 1:
                # Hidden layers
                lora_update = self.lora_layers[f"lora_{layer_idx}"](current)
                lora_contribution = lora_contribution + lora_update
                current = self.activation(module(current) + lora_update)
            else:
                # Output layer
                base_out = module(current)
                lora_out = self.lora_layers[f"lora_{layer_idx}"](current)
                return base_out + lora_out
                # lora_update = self.lora_layers[f"lora_{layer_idx}"](current)
                # lora_contribution = lora_contribution + lora_update
            
            layer_idx += 1
        
        # Return base output + LoRA contribution
        # return base_out + lora_contribution


print("LoRA model classes defined successfully!")

## Section 2: Load Pretrained Model and Wrap with LoRA

In [ ]:
# =========================================================================================
# Block 2-1: Define Original Model Architecture

class FeedforwardNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=1, num_hidden_layers=2):
        super().__init__()
        layers = []
        # Input layer
        layers.append(nn.Linear(in_dim, hidden_dim))
        layers.append(nn.SiLU())
        # Hidden layers
        for _ in range(num_hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.SiLU())
        # Output layer
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

print("Original model architecture defined!")

In [ ]:
# =========================================================================================
# Block 2-2: Load Pretrained Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create and load the original pretrained model
in_dim = X.shape[1]  # 15 features
hidden_dim = 256
out_dim = 1
num_hidden_layers = 5

pretrained_model = FeedforwardNN(in_dim, hidden_dim, out_dim, num_hidden_layers).to(device)
pretrained_model.load_state_dict(torch.load("pretrained_3C90.pth", map_location=device))
pretrained_model.eval()
print("Pretrained model loaded successfully!")

# Count parameters in pretrained model
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

pretrained_params = count_parameters(pretrained_model)
print(f"Pretrained model parameters: {pretrained_params}")

In [ ]:
# =========================================================================================
# Block 2-3: Create LoRA-enhanced Model

# LoRA hyperparameters
# lora_rank = 8  # Rank of LoRA matrices
lora_ranks = np.array([1, 2, 4, 8, 16, 32, 64, 128])  # Rank of LoRA matrices
lora_alpha = 16  # Scaling factor
lora_dropout = 0.1  # Dropout in LoRA layers

# Create LoRA model with different LoRA ranks
lora_models = np.empty(len(lora_ranks), dtype=object)

for i in range(len(lora_ranks)):
    lora_models[i] = FeedforwardNNWithLoRA(
        pretrained_model,
        rank=lora_ranks[i],
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout
    ).to(device)

# Count trainable parameters in LoRA model
# lora_params = count_parameters(lora_model)
# print(f"LoRA model trainable parameters: {lora_params}")
# print(f"Parameter reduction: {100 * (1 - lora_params / pretrained_params):.2f}%")
import pandas as pd

lora_params = np.empty(len(lora_ranks))
params_df = np.empty(len(lora_ranks))
for i in range(len(lora_ranks)):
    lora_params[i] = count_parameters(lora_models[i])
    params_df[i] = 100 * (1 - lora_params[i] / pretrained_params)

param_results = []
for i in range(len(lora_ranks)):
    param_results.append({"Rank": f"{lora_ranks[i]}", "Trainable Parameters": lora_params[i], "Parameter Reduction": params_df[i]})
params_summary = pd.DataFrame(param_results)
print("\n=== LoRA FINE-TUNING PARAMETER SUMMARY ====")
print(params_summary)

## Section 3: LoRA Fine-tuning
To be adjusted:
- Block 3-1:
    - Optimizer
    - Scheduler
- Block 3-2:
    - Epoch #

In [ ]:
# =========================================================================================
# Block 3-1: Setup Optimization

# Only optimize LoRA parameters
lora_params_lists = []
for i in range(len(lora_ranks)):
    lora_params_lists.append(list(lora_models[i].lora_layers.parameters()))

criterion = nn.MSELoss()

optimizers = []
schedulers = []
for i in range(len(lora_ranks)):
    optimizers.append(torch.optim.Adam(lora_params_lists[i], lr=1e-3))
    schedulers.append(torch.optim.lr_scheduler.CosineAnnealingLR(optimizers[i], T_max=100))

# print(f"Number of LoRA parameters to optimize: {sum(p.numel() for p in lora_params_list)}")
params2optimize = []
for i in range(len(lora_ranks)):
    params2optimize.append({"Rank": f"{lora_ranks[i]}", "Parameters to Optimize": sum(p.numel() for p in lora_params_lists[i])})
params2optimize_summary = pd.DataFrame(params2optimize)
print("\n=== LoRA FINE-TUNING PARAMETER SUMMARY ====")
print(params2optimize_summary)
print(f"Optimizer: Adam, lr=1e-3")
print(f"Scheduler: CosineAnnealingLR, T_max=100")

In [ ]:
# =========================================================================================
# Block 3-2-1: Training Loop

import copy

num_epochs = 150
best_val_losses = np.full(len(lora_ranks), float('inf'))
best_model_states = [None] * len(lora_ranks)
all_train_losses = []
all_val_losses = []

for i in range(len(lora_ranks)):
    print(f"\n=== Training LoRA model {i+1} (rank={lora_ranks[i]}) ====")
    
    # Current model, optimizer and scheduler
    current_model = lora_models[i]
    current_optimizer = optimizers[i]
    current_scheduler = schedulers[i]
    
    # Records of current model
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_model_state = None
    
    # Training loop
    for epoch in range(num_epochs):
        # Training phase
        lora_models[i].train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            current_optimizer.zero_grad()
            preds = current_model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            current_optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)
    
        # Validation phase
        current_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = current_model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)
    
        # Scheduler step
        current_scheduler.step()
    
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(current_model.state_dict())
    
        if (epoch + 1) % 25 == 0 or epoch == 0:
            current_lr = current_optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f} - LR: {current_lr:.6f}")

    # Load best model
    current_model.load_state_dict(best_model_state)
    best_val_losses[i] = best_val_loss
    best_model_states[i] = best_model_state
    all_train_losses.append(train_losses)
    all_val_losses.append(val_losses)
    print(f"Rank {lora_ranks[i]} - Best validation loss: {best_val_loss:.6f}")

## Section 4: Model Evaluation

In [ ]:
# =========================================================================================
# Block 4-1: Evaluate Model Accuracy

def evaluate_model_accuracy(loader, scaler_y, model, device, set_name="Set"):
    """
    Evaluates the regression performance of the model on a given dataset loader.
    """
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            preds = model(xb).cpu().numpy()
            y_pred.append(preds)
            y_true.append(yb.cpu().numpy())
    y_true = np.vstack(y_true)
    y_pred = np.vstack(y_pred)
    # Inverse transform to original y scale
    y_true_log = scaler_y.inverse_transform(y_true)
    y_pred_log = scaler_y.inverse_transform(y_pred)
    y_true_orig = np.exp(y_true_log)
    y_pred_orig = np.exp(y_pred_log)
    mae = np.mean(np.abs(y_true_orig - y_pred_orig))
    rmse = np.sqrt(np.mean((y_true_orig - y_pred_orig) ** 2))
    epsilon = 1e-8
    mape = np.mean(np.abs((y_true_orig - y_pred_orig) / (y_true_orig + epsilon))) * 100
    size = y_true_orig.shape[0]
    return mae, rmse, mape, size, y_true_orig, y_pred_orig

# Evaluate on all sets
all_train_mae = np.empty(len(lora_ranks))
all_train_rmse = np.empty(len(lora_ranks))
all_train_mape = np.empty(len(lora_ranks))
train_sizes = np.empty(len(lora_ranks))
all_y_train_true = np.empty(len(lora_ranks), dtype=object)
all_y_train_pred = np.empty(len(lora_ranks), dtype=object)
all_val_mae = np.empty(len(lora_ranks))
all_val_rmse = np.empty(len(lora_ranks))
all_val_mape = np.empty(len(lora_ranks))
val_sizes = np.empty(len(lora_ranks))
all_y_val_true = np.empty(len(lora_ranks), dtype=object)
all_y_val_pred = np.empty(len(lora_ranks), dtype=object)
all_test_mae = np.empty(len(lora_ranks))
all_test_rmse = np.empty(len(lora_ranks))
all_test_mape = np.empty(len(lora_ranks))
test_sizes = np.empty(len(lora_ranks))
all_y_test_true = np.empty(len(lora_ranks), dtype=object)
all_y_test_pred = np.empty(len(lora_ranks), dtype=object)

for i in range(len(lora_ranks)):
    all_train_mae[i], all_train_rmse[i], all_train_mape[i], train_sizes[i], all_y_train_true[i], all_y_train_pred[i] = evaluate_model_accuracy(
        train_loader, output_scaler, lora_models[i], device, set_name="Train Set"
    )
    all_val_mae[i], all_val_rmse[i], all_val_mape[i], val_sizes[i], all_y_val_true[i], all_y_val_pred[i] = evaluate_model_accuracy(
        val_loader, output_scaler, lora_models[i], device, set_name="Validation Set"
    )
    all_test_mae[i], all_test_rmse[i], all_test_mape[i], test_sizes[i], all_y_test_true[i], all_y_test_pred[i] = evaluate_model_accuracy(
        test_loader, output_scaler, lora_models[i], device, set_name="Test Set"
    )

# Print results
import pandas as pd
results = []
for i in range(len(lora_ranks)):
    results.append([
        {"Set": "Train", "Size": train_sizes[i], "MAE": all_train_mae[i], "RMSE": all_train_rmse[i], "MAPE (%)": all_train_mape[i]},
        {"Set": "Validation", "Size": val_sizes[i], "MAE": all_val_mae[i], "RMSE": all_val_rmse[i], "MAPE (%)": all_val_mape[i]},
        {"Set": "Test", "Size": test_sizes[i], "MAE": all_test_mae[i], "RMSE": all_test_rmse[i], "MAPE (%)": all_test_mape[i]}
    ])
    metrics_df = pd.DataFrame(results[i])
    print(f"\n=== LoRA FINE-TUNING METRICS SUMMARY WHEN RANK={lora_ranks[i]}====")
    print(metrics_df)

In [ ]:
# =========================================================================================
# Block 4-2: Plot and Visualize Results

import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 12

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.figure(figsize=(7, 5))
# plt.plot(lora_params, all_test_mae, marker='o')
plt.scatter(lora_params, all_test_mae)
for x, y, r in zip(lora_params, all_test_mae, lora_ranks):
    plt.annotate(
        f"$r=${r}",
        (x, y),
        textcoords="offset points",
        xytext=(5, 5),
        ha='left',
        fontsize=10
    )
plt.xlabel("Number of Trainable Parameters")
plt.ylabel("Test MAE")
plt.title("Parameter Efficiency of LoRA")
plt.grid()
plt.tight_layout()

plt.savefig('plots/rank_comparison_plot_77.svg')